# Clase 16 — Casos de Uso

Este notebook desarrolla los **casos de uso** propuestos en:

- **Sesión 1: Repaso `combineByKey`** → 🧪 *Análisis de Actividad de Usuarios en una Plataforma Digital* (e-learning)
- **Sesión 2: Introducción a DataFrames** → 🌾 *AgroData Cooperativa*

Entorno: Apache Spark 4.1.1 + Scala 2.13 (Almond kernel) en modo `local[*]`.

## 0. Inicialización de Spark

In [1]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`
import org.apache.log4j.{Level, Logger}
Logger.getLogger("org").setLevel(Level.ERROR)
Logger.getLogger("akka").setLevel(Level.ERROR)

import org.apache.spark.sql.SparkSession

val spark = SparkSession.builder()
  .appName("Clase16-CasosDeUso")
  .master("local[*]")
  .config("spark.ui.showConsoleProgress", "false")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
import spark.implicits._
val sc = spark.sparkContext

println(s"✅ Spark ${spark.version} iniciado en ${sc.master}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/30 07:39:12 INFO SparkContext: Running Spark version 4.1.1
26/04/30 07:39:12 INFO SparkContext: OS info Windows 11, 10.0, amd64
26/04/30 07:39:12 INFO SparkContext: Java version 17.0.18+8
26/04/30 07:39:13 INFO ResourceUtils: ==============================================================
26/04/30 07:39:13 INFO ResourceUtils: No custom resources configured for spark.driver.
26/04/30 07:39:13 INFO ResourceUtils: ==============================================================
26/04/30 07:39:13 INFO SparkContext: Submitted application: Clase16-CasosDeUso
26/04/30 07:39:13 INFO SecurityManager: Changing view acls to: gre
26/04/30 07:39:13 INFO SecurityManager: Changing modify acls to: gre
26/04/30 07:39:13 INFO SecurityManager: Changing view acls groups to: gre
26/04/30 07:39:13 INFO SecurityManager: Changing modify acls groups to: gre
26/04/30 07:39:13 INFO SecurityManager: SecurityManager: authentication

✅ Spark 4.1.1 iniciado en local[*]


import $ivy.$
import org.apache.log4j.{Level, Logger}
import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@26f04254
import spark.implicits._
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@6954195b

---

# 🧪 Caso de Uso — Sesión 1: Análisis de Actividad de Usuarios

## 🏢 Contexto

Una empresa de e-learning quiere analizar el comportamiento de sus usuarios. Cada acción del usuario en la plataforma genera un registro `(usuario, tipo_evento, duracion_minutos)`. El objetivo es construir, con `combineByKey`, una estructura `usuario → List[duraciones]` y a partir de ella calcular el **tiempo total** y el **tiempo medio** por usuario.

## 📥 Datos de entrada

In [2]:
val eventos = sc.parallelize(List(
  ("user1", "video", 10.0),
  ("user2", "quiz",   5.0),
  ("user1", "video", 15.0),
  ("user3", "video", 20.0),
  ("user2", "video",  8.0),
  ("user1", "quiz",   7.0),
  ("user3", "quiz",   6.0),
  ("user2", "video", 12.0),
  ("user1", "video",  9.0),
  ("user3", "video", 11.0),
  ("user2", "quiz",   4.0),
  ("user3", "video", 13.0)
))

println(s"Eventos cargados: ${eventos.count()}")
println(s"Particiones:      ${eventos.getNumPartitions}")

Eventos cargados: 12
Particiones:      16


eventos: org.apache.spark.rdd.RDD[(String, String, Double)] = ParallelCollectionRDD[0] at parallelize at cmd2.sc:1

## 🔍 Paso 1 — Visualizar particiones

Definimos una pequeña utilidad `verParticiones` (mencionada en el enunciado) y la aplicamos al RDD.

In [3]:
import org.apache.spark.rdd.RDD

def verParticiones[T](rdd: RDD[T]): Unit = {
  val datos = rdd.glom().collect()
  println(s"Número de particiones: ${datos.length}")
  datos.zipWithIndex.foreach { case (parte, i) =>
    println(s"  Partición $i (${parte.length} elementos): ${parte.mkString(", ")}")
  }
}

verParticiones(eventos)

Número de particiones: 16
  Partición 0 (0 elementos): 
  Partición 1 (1 elementos): (user1,video,10.0)
  Partición 2 (1 elementos): (user2,quiz,5.0)
  Partición 3 (1 elementos): (user1,video,15.0)
  Partición 4 (0 elementos): 
  Partición 5 (1 elementos): (user3,video,20.0)
  Partición 6 (1 elementos): (user2,video,8.0)
  Partición 7 (1 elementos): (user1,quiz,7.0)
  Partición 8 (0 elementos): 
  Partición 9 (1 elementos): (user3,quiz,6.0)
  Partición 10 (1 elementos): (user2,video,12.0)
  Partición 11 (1 elementos): (user1,video,9.0)
  Partición 12 (0 elementos): 
  Partición 13 (1 elementos): (user3,video,11.0)
  Partición 14 (1 elementos): (user2,quiz,4.0)
  Partición 15 (1 elementos): (user3,video,13.0)


import org.apache.spark.rdd.RDD
defined function verParticiones

## 1️⃣ Transformar el dataset a `(usuario, duracion)`

In [4]:
val eventosPorUsuario = eventos.map { case (user, _, duracion) =>
  (user, duracion)
}

println("Pair RDD (usuario, duracion):")
eventosPorUsuario.collect().foreach(println)

Pair RDD (usuario, duracion):
(user1,10.0)
(user2,5.0)
(user1,15.0)
(user3,20.0)
(user2,8.0)
(user1,7.0)
(user3,6.0)
(user2,12.0)
(user1,9.0)
(user3,11.0)
(user2,4.0)
(user3,13.0)


eventosPorUsuario: RDD[(String, Double)] = MapPartitionsRDD[2] at map at cmd4.sc:1

## 2️⃣ Aplicar `combineByKey` para construir `usuario → List[duraciones]`

- **createCombiner**: cuando aparece el primer valor de una clave en una partición, lo envolvemos en una lista de un solo elemento.
- **mergeValue**: dentro de la misma partición, añadimos cada nuevo valor a la lista existente.
- **mergeCombiners**: al hacer el shuffle, fusionamos las listas que cada partición construyó por separado.

In [5]:
val resultado = eventosPorUsuario.combineByKey(
  (d: Double)               => List(d),               // createCombiner
  (acc: List[Double], d: Double)        => acc :+ d,  // mergeValue (dentro de partición)
  (a: List[Double], b: List[Double])    => a ++ b      // mergeCombiners (entre particiones)
)

println("=== Resultado: usuario → List[duraciones] ===")
resultado.collect().sortBy(_._1).foreach { case (u, lista) =>
  println(s"  $u → List(${lista.mkString(", ")})")
}

=== Resultado: usuario → List[duraciones] ===
  user1 → List(10.0, 15.0, 7.0, 9.0)
  user2 → List(5.0, 8.0, 12.0, 4.0)
  user3 → List(20.0, 6.0, 11.0, 13.0)


resultado: RDD[(String, List[Double])] = ShuffledRDD[3] at combineByKey at cmd5.sc:4

## 🚀 Extensión

### 🔹 Tiempo total por usuario

In [6]:
val totalPorUsuario = resultado.mapValues(_.sum)

println("=== Tiempo TOTAL por usuario ===")
totalPorUsuario.collect().sortBy(_._1).foreach { case (u, total) =>
  println(f"  $u → $total%.1f min")
}

=== Tiempo TOTAL por usuario ===
  user1 → 41,0 min
  user2 → 29,0 min
  user3 → 50,0 min


totalPorUsuario: RDD[(String, Double)] = MapPartitionsRDD[4] at mapValues at cmd6.sc:1

### 🔹 Tiempo medio por usuario

In [7]:
val mediaPorUsuario = resultado.mapValues(lista => lista.sum / lista.length)

println("=== Tiempo MEDIO por usuario ===")
mediaPorUsuario.collect().sortBy(_._1).foreach { case (u, media) =>
  println(f"  $u → $media%.2f min/evento")
}

=== Tiempo MEDIO por usuario ===
  user1 → 10,25 min/evento
  user2 → 7,25 min/evento
  user3 → 12,50 min/evento


mediaPorUsuario: RDD[(String, Double)] = MapPartitionsRDD[5] at mapValues at cmd7.sc:1

## 💡 Preguntas para reflexión

**1. ¿Qué ocurre dentro de cada partición?**

Spark recorre los pares `(clave, valor)` localmente. La primera vez que ve una clave nueva en esa partición ejecuta `createCombiner` para inicializar el acumulador (en nuestro caso, `List(d)`). Para cada valor adicional de la misma clave aplica `mergeValue`, que añade el elemento a la lista. Todo esto sucede **sin movimiento de datos por la red**: es procesamiento puramente local.

**2. ¿Cuándo se ejecuta `mergeCombiners`?**

Se ejecuta **después del shuffle**, cuando los acumuladores parciales producidos en distintas particiones para la misma clave se reúnen en el mismo nodo reducer. Spark fusiona esos acumuladores entre sí con `mergeCombiners` (en nuestro caso, concatenando las listas con `++`).

**3. ¿Por qué `combineByKey` es más flexible que `reduceByKey`?**

Porque permite que **el tipo del acumulador sea distinto al tipo del valor de entrada**. Aquí el valor de entrada es `Double` pero el acumulador es `List[Double]`. `reduceByKey` exige que entrada y salida sean del mismo tipo, así que no podría construir una lista directamente. `combineByKey` también permite controlar explícitamente cómo se inicializa el acumulador para cada clave nueva.

**4. ¿Qué pasaría si usas `groupByKey` en este caso?**

Funcionaría y daría un resultado equivalente (`(usuario, Iterable[Double])`), pero sería **mucho menos eficiente**: `groupByKey` envía **todos los valores en bruto** por la red sin precombinar nada en cada partición. Con datasets grandes esto provoca shuffles costosos y posibles `OutOfMemoryError` en el reducer. `combineByKey` (al igual que `reduceByKey` y `aggregateByKey`) **precombina localmente** antes del shuffle, reduciendo drásticamente el volumen de datos transmitido.

---

# 🌾 Caso de Uso — Sesión 2: AgroData Cooperativa

Cooperativa agrícola con productores en cinco provincias españolas. Centralizamos los datos exportados en CSV (parcelas) y JSON (catálogo de productos) para extraer un primer informe de situación.

## 🛠️ Generación de los ficheros de datos

Generamos `parcelas.csv` y `productos.json` desde Scala en `C:\Curso-Scala\datos\agrodata\`.

In [8]:
import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets

val rutaBase = Paths.get("C:/Curso-Scala/datos/agrodata")
Files.createDirectories(rutaBase)

val parcelasCsv =
  """id_parcela,provincia,municipio,superficie_ha,cultivo,año_alta,en_produccion,rendimiento_kg_ha
    |P001,Sevilla,Carmona,12.5,Naranja,2015,true,28000
    |P002,Huelva,Lepe,8.3,Fresa,2018,true,45000
    |P003,Almería,Níjar,25.0,Tomate,2012,true,85000
    |P004,Sevilla,Écija,6.7,Aceituna,2009,false,3200
    |P005,Murcia,Totana,15.2,Limón,2016,true,22000
    |P006,Almería,El Ejido,30.1,Pimiento,2014,true,62000
    |P007,Huelva,Moguer,9.8,Fresa,2020,true,41000
    |P008,Murcia,Lorca,18.4,Melocotón,2011,false,9500
    |P009,Sevilla,Utrera,22.0,Naranja,2013,true,31000
    |P010,Almería,Vícar,11.6,Pepino,2019,true,74000
    |P011,Murcia,Alhama,7.9,Limón,2017,true,20500
    |P012,Huelva,Cartaya,14.3,Fresa,2015,true,43000
    |P013,Sevilla,Marchena,5.2,Aceituna,2010,false,2900
    |P014,Almería,Roquetas,28.7,Tomate,2011,true,88000
    |P015,Murcia,Mazarrón,16.5,Pimiento,2018,true,58000
    |""".stripMargin

val productosJson =
  """[
    |  {"codigo":"NAR","nombre":"Naranja","familia":"Citrico","precio_mercado_euro_kg":0.45,"demanda_exportacion":"Alta","certificacion_eco":false},
    |  {"codigo":"FRE","nombre":"Fresa","familia":"Baya","precio_mercado_euro_kg":2.10,"demanda_exportacion":"Muy Alta","certificacion_eco":true},
    |  {"codigo":"TOM","nombre":"Tomate","familia":"Hortaliza","precio_mercado_euro_kg":0.85,"demanda_exportacion":"Alta","certificacion_eco":false},
    |  {"codigo":"ACE","nombre":"Aceituna","familia":"Oleaginosa","precio_mercado_euro_kg":0.60,"demanda_exportacion":"Media","certificacion_eco":true},
    |  {"codigo":"LIM","nombre":"Limón","familia":"Citrico","precio_mercado_euro_kg":0.55,"demanda_exportacion":"Alta","certificacion_eco":false},
    |  {"codigo":"PIM","nombre":"Pimiento","familia":"Hortaliza","precio_mercado_euro_kg":1.20,"demanda_exportacion":"Muy Alta","certificacion_eco":true},
    |  {"codigo":"PEP","nombre":"Pepino","familia":"Hortaliza","precio_mercado_euro_kg":0.70,"demanda_exportacion":"Media","certificacion_eco":false},
    |  {"codigo":"MEL","nombre":"Melocotón","familia":"Drupa","precio_mercado_euro_kg":1.35,"demanda_exportacion":"Media","certificacion_eco":false}
    |]
    |""".stripMargin

Files.write(rutaBase.resolve("parcelas.csv"), parcelasCsv.getBytes(StandardCharsets.UTF_8))
Files.write(rutaBase.resolve("productos.json"), productosJson.getBytes(StandardCharsets.UTF_8))

println(s"✅ Ficheros creados en: ${rutaBase.toAbsolutePath}")

✅ Ficheros creados en: C:\Curso-Scala\datos\agrodata


import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets
rutaBase: java.nio.file.Path = C:\Curso-Scala\datos\agrodata
res8_3: java.nio.file.Path = C:\Curso-Scala\datos\agrodata
parcelasCsv: String = """id_parcela,provincia,municipio,superficie_ha,cultivo,año_alta,en_produccion,rendimiento_kg_ha
P001,Sevilla,Carmona,12.5,Naranja,2015,true,28000
P002,Huelva,Lepe,8.3,Fresa,2018,true,45000
P003,Almería,Níjar,25.0,Tomate,2012,true,85000
P004,Sevilla,Écija,6.7,Aceituna,2009,false,3200
P005,Murcia,Totana,15.2,Limón,2016,true,22000
P006,Almería,El Ejido,30.1,Pimiento,2014,true,62000
P007,Huelva,Moguer,9.8,Fresa,2020,true,41000
P008,Murcia,Lorca,18.4,Melocotón,2011,false,9500
P009,Sevilla,Utrera,22.0,Naranja,2013,true,31000
P010,Almería,Vícar,11.6,Pepino,2019,true,74000
P011,Murcia,Alhama,7.9,Limón,2017,true,20500
P012,Huelva,Cartaya,14.3,Fresa,2015,true,43000
P013,Sevilla,Marchena,5.2,Aceituna,2010,false,2900
P014,Almería,Roquetas,28.7,Tomate,2011,true,88000
P015,Murcia,Maz

## 📋 Tarea 1 — Inicialización del entorno

La `SparkSession` ya se creó en la celda 0. Aquí confirmamos imports adicionales y la versión.

In [9]:
import org.apache.spark.sql.types._
// spark.implicits._ ya importado en la celda 0

println(s"✅ AgroData Analytics iniciado — Spark ${spark.version}")

✅ AgroData Analytics iniciado — Spark 4.1.1


import org.apache.spark.sql.types._

## 📋 Tarea 2 — Carga del CSV de parcelas con `inferSchema`

In [10]:
val dfParcelasInf = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("C:/Curso-Scala/datos/agrodata/parcelas.csv")

println("=== Primeras 5 parcelas ===")
dfParcelasInf.show(5)

println("=== Schema inferido ===")
dfParcelasInf.printSchema()

println(s"Total de parcelas registradas: ${dfParcelasInf.count()}")
println(s"Columnas: ${dfParcelasInf.columns.mkString(" | ")}")

=== Primeras 5 parcelas ===
+----------+---------+---------+-------------+--------+--------+-------------+-----------------+
|id_parcela|provincia|municipio|superficie_ha| cultivo|año_alta|en_produccion|rendimiento_kg_ha|
+----------+---------+---------+-------------+--------+--------+-------------+-----------------+
|      P001|  Sevilla|  Carmona|         12.5| Naranja|    2015|         true|            28000|
|      P002|   Huelva|     Lepe|          8.3|   Fresa|    2018|         true|            45000|
|      P003|  Almería|    Níjar|         25.0|  Tomate|    2012|         true|            85000|
|      P004|  Sevilla|    Écija|          6.7|Aceituna|    2009|        false|             3200|
|      P005|   Murcia|   Totana|         15.2|   Limón|    2016|         true|            22000|
+----------+---------+---------+-------------+--------+--------+-------------+-----------------+
only showing top 5 rows
=== Schema inferido ===
root
 |-- id_parcela: string (nullable = true)
 |--

dfParcelasInf: org.apache.spark.sql.package.DataFrame = [id_parcela: string, provincia: string ... 6 more fields]

## 📋 Tarea 3 — Definir el schema manualmente y recargar

Correcciones requeridas:
- `id_parcela`: `nullable = false` (clave que nunca debe ser nula).
- `rendimiento_kg_ha`: `DoubleType` en lugar de `IntegerType` para permitir cálculos con decimales.

In [11]:
val schemaParcelas = StructType(Array(
  StructField("id_parcela",        StringType,  nullable = false),
  StructField("provincia",         StringType,  nullable = true),
  StructField("municipio",         StringType,  nullable = true),
  StructField("superficie_ha",     DoubleType,  nullable = true),
  StructField("cultivo",           StringType,  nullable = true),
  StructField("año_alta",          IntegerType, nullable = true),
  StructField("en_produccion",     BooleanType, nullable = true),
  StructField("rendimiento_kg_ha", DoubleType,  nullable = true)
))

val dfParcelas = spark.read
  .option("header", "true")
  .schema(schemaParcelas)
  .csv("C:/Curso-Scala/datos/agrodata/parcelas.csv")

println("=== Schema manual aplicado ===")
dfParcelas.printSchema()

println("=== Tipos por columna ===")
dfParcelas.dtypes.foreach { case (col, tipo) =>
  println(f"  ${col}%-18s → $tipo")
}

=== Schema manual aplicado ===
root
 |-- id_parcela: string (nullable = true)
 |-- provincia: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- superficie_ha: double (nullable = true)
 |-- cultivo: string (nullable = true)
 |-- año_alta: integer (nullable = true)
 |-- en_produccion: boolean (nullable = true)
 |-- rendimiento_kg_ha: double (nullable = true)

=== Tipos por columna ===
  id_parcela         → StringType
  provincia          → StringType
  municipio          → StringType
  superficie_ha      → DoubleType
  cultivo            → StringType
  año_alta           → IntegerType
  en_produccion      → BooleanType
  rendimiento_kg_ha  → DoubleType


schemaParcelas: StructType = Seq(
  StructField(
    name = "id_parcela",
    dataType = StringType,
    nullable = false,
    metadata = {}
  ),
  StructField(
    name = "provincia",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "municipio",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "superficie_ha",
    dataType = DoubleType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "cultivo",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "año_alta",
    dataType = IntegerType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "en_produccion",
...
dfParcelas: org.apache.spark.sql.package.DataFrame = [id_parcela: string, provincia: string ... 6 more fields]

## 📋 Tarea 4 — Primer informe estadístico de las parcelas

In [12]:
println("=== Estadísticas de superficie y rendimiento ===")
dfParcelas.describe("superficie_ha", "rendimiento_kg_ha").show()

println("=== Rango de años de alta ===")
dfParcelas.describe("año_alta").show()

=== Estadísticas de superficie y rendimiento ===
+-------+------------------+------------------+
|summary|     superficie_ha| rendimiento_kg_ha|
+-------+------------------+------------------+
|  count|                15|                15|
|   mean|15.479999999999999|40873.333333333336|
| stddev| 7.931240220077275|27911.479120008433|
|    min|               5.2|            2900.0|
|    max|              30.1|           88000.0|
+-------+------------------+------------------+

=== Rango de años de alta ===
+-------+------------------+
|summary|          año_alta|
+-------+------------------+
|  count|                15|
|   mean|2014.5333333333333|
| stddev|3.4613512362879963|
|    min|              2009|
|    max|              2020|
+-------+------------------+



## 📋 Tarea 5 — Carga del catálogo de productos en JSON

In [13]:
val dfProductosInf = spark.read
  .option("multiline", "true")
  .json("C:/Curso-Scala/datos/agrodata/productos.json")

println("=== Catálogo de productos ===")
dfProductosInf.show(truncate = false)

println("=== Schema del JSON ===")
dfProductosInf.printSchema()

// Anotación sobre los tipos inferidos:
//   - precio_mercado_euro_kg → DoubleType (correcto: precios decimales)
//   - certificacion_eco      → BooleanType (correcto: true/false en el JSON)
//   - Spark ordena las columnas alfabéticamente, comportamiento normal del lector JSON.
println("📌 precio_mercado_euro_kg → DoubleType  ✅")
println("📌 certificacion_eco      → BooleanType ✅")
println("📌 Las columnas aparecen en orden alfabético (lector JSON de Spark).")

=== Catálogo de productos ===
+-----------------+------+-------------------+----------+---------+----------------------+
|certificacion_eco|codigo|demanda_exportacion|familia   |nombre   |precio_mercado_euro_kg|
+-----------------+------+-------------------+----------+---------+----------------------+
|false            |NAR   |Alta               |Citrico   |Naranja  |0.45                  |
|true             |FRE   |Muy Alta           |Baya      |Fresa    |2.1                   |
|false            |TOM   |Alta               |Hortaliza |Tomate   |0.85                  |
|true             |ACE   |Media              |Oleaginosa|Aceituna |0.6                   |
|false            |LIM   |Alta               |Citrico   |Limón    |0.55                  |
|true             |PIM   |Muy Alta           |Hortaliza |Pimiento |1.2                   |
|false            |PEP   |Media              |Hortaliza |Pepino   |0.7                   |
|false            |MEL   |Media              |Drupa     |Mel

dfProductosInf: org.apache.spark.sql.package.DataFrame = [certificacion_eco: boolean, codigo: string ... 4 more fields]

## 📋 Tarea 6 — Schema manual para el catálogo de productos

Restricciones:
- `codigo`: `nullable = false` (clave del producto).
- `precio_mercado_euro_kg`: `DoubleType` con `nullable = false`.

In [14]:
val schemaProductos = StructType(Array(
  StructField("codigo",                 StringType,  nullable = false),
  StructField("nombre",                 StringType,  nullable = true),
  StructField("familia",                StringType,  nullable = true),
  StructField("precio_mercado_euro_kg", DoubleType,  nullable = false),
  StructField("demanda_exportacion",    StringType,  nullable = true),
  StructField("certificacion_eco",      BooleanType, nullable = true)
))

val dfProductos = spark.read
  .option("multiline", "true")
  .schema(schemaProductos)
  .json("C:/Curso-Scala/datos/agrodata/productos.json")

println("=== Schema manual del catálogo ===")
dfProductos.printSchema()

println("=== Comparativa de tipos: inferido vs manual ===")
val tiposInf    = dfProductosInf.dtypes.toMap
val tiposManual = dfProductos.dtypes.toMap
val todasLasCols = (tiposInf.keySet ++ tiposManual.keySet).toSeq.sorted
println(f"  ${"Columna"}%-25s ${"Inferido"}%-15s ${"Manual"}%-15s")
println("  " + "-" * 55)
todasLasCols.foreach { c =>
  val a = tiposInf.getOrElse(c, "-")
  val b = tiposManual.getOrElse(c, "-")
  println(f"  ${c}%-25s ${a}%-15s ${b}%-15s")
}

=== Schema manual del catálogo ===
root
 |-- codigo: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- familia: string (nullable = true)
 |-- precio_mercado_euro_kg: double (nullable = true)
 |-- demanda_exportacion: string (nullable = true)
 |-- certificacion_eco: boolean (nullable = true)

=== Comparativa de tipos: inferido vs manual ===
  Columna                   Inferido        Manual         
  -------------------------------------------------------
  certificacion_eco         BooleanType     BooleanType    
  codigo                    StringType      StringType     
  demanda_exportacion       StringType      StringType     
  familia                   StringType      StringType     
  nombre                    StringType      StringType     
  precio_mercado_euro_kg    DoubleType      DoubleType     


schemaProductos: StructType = Seq(
  StructField(
    name = "codigo",
    dataType = StringType,
    nullable = false,
    metadata = {}
  ),
  StructField(
    name = "nombre",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "familia",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "precio_mercado_euro_kg",
    dataType = DoubleType,
    nullable = false,
    metadata = {}
  ),
  StructField(
    name = "demanda_exportacion",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "certificacion_eco",
    dataType = BooleanType,
    nullable = true,
    metadata = {}
  )
)
dfProductos: org.apache.spark.sql.package.DataFrame = [codigo: string, nombre: string ... 4 more fields]
tiposInf: Map[String, String] = HashMap(
  "codigo" -> "StringType",
  "certificacion_eco" -> "BooleanType",
  "familia" -> "StringType",
  "precio_mercado_euro_kg" -> "Doub

## 📋 Tarea 7 — DataFrame de resumen desde colección en memoria

In [15]:
val resumenProvincias = Seq(
  ("Almería", 4, 95.4, 4),
  ("Huelva",  3, 32.4, 3),
  ("Murcia",  4, 58.0, 3),
  ("Sevilla", 4, 46.4, 2)
).toDF("provincia", "num_parcelas", "superficie_total_ha", "parcelas_activas")

println("=== Resumen por provincia ===")
resumenProvincias.show()

println("=== Schema del resumen ===")
resumenProvincias.printSchema()

println(s"Filas:    ${resumenProvincias.count()}")
println(s"Columnas: ${resumenProvincias.columns.mkString(" | ")}")

=== Resumen por provincia ===
+---------+------------+-------------------+----------------+
|provincia|num_parcelas|superficie_total_ha|parcelas_activas|
+---------+------------+-------------------+----------------+
|  Almería|           4|               95.4|               4|
|   Huelva|           3|               32.4|               3|
|   Murcia|           4|               58.0|               3|
|  Sevilla|           4|               46.4|               2|
+---------+------------+-------------------+----------------+

=== Schema del resumen ===
root
 |-- provincia: string (nullable = true)
 |-- num_parcelas: integer (nullable = false)
 |-- superficie_total_ha: double (nullable = false)
 |-- parcelas_activas: integer (nullable = false)

Filas:    4
Columnas: provincia | num_parcelas | superficie_total_ha | parcelas_activas


resumenProvincias: org.apache.spark.sql.package.DataFrame = [provincia: string, num_parcelas: int ... 2 more fields]

---

## 🛑 Cerrar la sesión de Spark (opcional)

In [15]:
// spark.stop()